In [12]:
# ============================================================
# RQ3 Obs 3.1 from COMMITS using intent_label_str
# - Explodes multi-label rows (e.g., "A || B") into multiple counts
# - Prints to notebook output AND writes the same log to a file
# ============================================================

from pathlib import Path
import pandas as pd
from datetime import datetime

try:
    from IPython.display import display
except Exception:
    display = print  # fallback


# -----------------------------
# CONFIG (edit if needed)
# -----------------------------
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output")
COMMITS_FILE = "1_All_Commits_PR_Msg_Iss_with_intentions_subcat_tfidf.csv"

# Log file name (auto-timestamped) - saved to BASE_DIR (or same folder as CSV)
LOG_BASENAME = "rq3_obs3_1_table5_commits_label_str_log"

# Filters
EXCLUDE_QA = True               # exclude rows where qa_issue is not null
ONLY_EVENT_TYPE_ADDED = True    # only adoption boundary events ("added")
REQUIRE_NONEMPTY_INTENT_LABEL_STR = True  # ensure intent_label_str is not empty
REQUIRE_INTENT_TOTAL_SCORE_GT0 = False    # optional extra guard

STYLE_RENAME = {
    "Emu_Community": "Community",
    "Emu_Custom": "Custom",
    "ThirdParty": "Third Party",
    "GMD": "GMD",
}

INTENTION_ORDER = [
    "Expand test scope or capabilities",
    "Introduce / strengthen CI-backed tests",
    "Address performance or stability issues",
    "Clean up or simplify CI / environment configuration",
    "Automate or integrate release workflows",
    "Migrate or modernise CI infrastructure",
]


# -----------------------------
# Logging (screen + file)
# -----------------------------
_log_path = None

def resolve_commits_path(base_dir: Path, commits_file: str) -> Path:
    candidate = base_dir / commits_file
    if candidate.exists():
        return candidate
    fallback = Path("/mnt/data") / commits_file  # fallback in this environment
    if fallback.exists():
        return fallback
    raise FileNotFoundError(f"Could not find commits CSV at:\n - {candidate}\n - {fallback}")

def init_log(out_dir: Path):
    global _log_path
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    _log_path = out_dir / f"{LOG_BASENAME}_{ts}.txt"
    _log_path.write_text("", encoding="utf-8")
    return _log_path

def log(msg=""):
    print(msg)
    if _log_path is not None:
        with _log_path.open("a", encoding="utf-8") as f:
            f.write(str(msg) + "\n")

def log_df(df: pd.DataFrame, title: str = None, max_rows: int = 80):
    if title:
        log(f"\n{title}")
    display(df)
    with pd.option_context("display.max_rows", max_rows, "display.max_columns", None, "display.width", 220):
        log(df.to_string())


# -----------------------------
# Helpers
# -----------------------------
def normalize_style(s):
    return STYLE_RENAME.get(s, s)

def split_labels(label_str):
    """
    Splits multi-label strings into labels.
    Expected: "Label A || Label B"
    Also supports ";" as fallback.
    """
    if pd.isna(label_str):
        return []
    s = str(label_str).strip()
    if not s or s.lower() == "nan":
        return []
    if "||" in s:
        parts = [p.strip() for p in s.split("||")]
    elif ";" in s:
        parts = [p.strip() for p in s.split(";")]
    else:
        parts = [s]
    return [p for p in parts if p]

def build_table5(long_df: pd.DataFrame) -> pd.DataFrame:
    vc = long_df["intention"].value_counts()
    total = int(vc.sum())
    tab = pd.DataFrame({"k": vc.astype(int), "pct": (vc / total * 100).round(1)})

    idx = [x for x in INTENTION_ORDER if x in tab.index] + [x for x in tab.index if x not in INTENTION_ORDER]
    tab = tab.loc[idx].copy()

    tab.loc["Total", :] = [total, 100.0]
    return tab


# -----------------------------
# RUN
# -----------------------------
commits_path = resolve_commits_path(BASE_DIR, COMMITS_FILE)
out_dir = commits_path.parent
log_path = init_log(out_dir)

# log("============================================================")
# log("RQ3 Observation 3.1 (Table 5) — COMMITS + intent_label_str")
# log("============================================================")
# log(f"Commits CSV: {commits_path}")
# log(f"Log file   : {log_path}")
# log("")
# log(f"EXCLUDE_QA={EXCLUDE_QA}, ONLY_EVENT_TYPE_ADDED={ONLY_EVENT_TYPE_ADDED}")
# log(f"REQUIRE_NONEMPTY_INTENT_LABEL_STR={REQUIRE_NONEMPTY_INTENT_LABEL_STR}")
# log(f"REQUIRE_INTENT_TOTAL_SCORE_GT0={REQUIRE_INTENT_TOTAL_SCORE_GT0}")

df = pd.read_csv(commits_path)
log(f"\nLoaded commits rows: {len(df)}")

# Required columns
for c in ["intent_label_str", "env_style"]:
    if c not in df.columns:
        raise KeyError(f"Missing required column: {c}")

f = df.copy()

# Filters
if EXCLUDE_QA and "qa_issue" in f.columns:
    before = len(f)
    f = f[f["qa_issue"].isna()].copy()
    log(f"After EXCLUDE_QA: {len(f)} (dropped {before - len(f)})")

if ONLY_EVENT_TYPE_ADDED:
    if "event_type" not in f.columns:
        raise KeyError("Missing column: event_type")
    before = len(f)
    f = f[f["event_type"].astype(str).str.lower().str.strip() == "added"].copy()
    log(f"After ONLY_EVENT_TYPE_ADDED: {len(f)} (dropped {before - len(f)})")

if REQUIRE_INTENT_TOTAL_SCORE_GT0:
    if "intent_total_score" not in f.columns:
        raise KeyError("REQUIRE_INTENT_TOTAL_SCORE_GT0=True but missing column: intent_total_score")
    before = len(f)
    f = f[f["intent_total_score"] > 0].copy()
    log(f"After REQUIRE_INTENT_TOTAL_SCORE_GT0: {len(f)} (dropped {before - len(f)})")

if REQUIRE_NONEMPTY_INTENT_LABEL_STR:
    before = len(f)
    s = f["intent_label_str"].astype(str).str.strip()
    f = f[s.ne("") & s.str.lower().ne("nan")].copy()
    log(f"After REQUIRE_NONEMPTY_INTENT_LABEL_STR: {len(f)} (dropped {before - len(f)})")

# Normalize style (not required for Table 5, but useful for debugging)
f["env_style"] = f["env_style"].map(normalize_style)

# Explode labels
rows = []
labels_per_row = []
for rid, r in f.iterrows():
    labs = split_labels(r["intent_label_str"])
    labels_per_row.append(len(labs))
    for lab in labs:
        rows.append({"row_id": rid, "env_style": r["env_style"], "intention": lab})

long_df = pd.DataFrame(rows)
if long_df.empty:
    raise RuntimeError("No labels found after filtering/exploding intent_label_str.")

labels_per_row = pd.Series(labels_per_row, index=f.index)
multi_rows = int((labels_per_row >= 2).sum())

log("\n=== Explosion sanity checks ===")
log(f"Commit rows contributing: {len(f)}")
log(f"Total counted intentions (sum k): {len(long_df)}")
log(f"Rows with 2+ labels: {multi_rows} ({(multi_rows/len(f)*100):.1f}%)")
log(f"Avg labels per row: {labels_per_row.mean():.3f}")

# Table 5
table5 = build_table5(long_df)
top2 = [x for x in INTENTION_ORDER[:2] if x in table5.index]
top2_share = float(table5.loc[top2, "pct"].sum()) if top2 else float("nan")

log("\n=== Observation 3.1 (Table 5) result ===")
log(f"Total counted intentions (Table 5 Total k): {int(table5.loc['Total', 'k'])}")
log(f"Top-2 intentions share: {top2_share:.1f}%")
log_df(table5, title="Table 5 (k and %)")

log("\nDONE.")
log("============================================================")



Loaded commits rows: 535
After EXCLUDE_QA: 524 (dropped 11)
After ONLY_EVENT_TYPE_ADDED: 504 (dropped 20)
After REQUIRE_NONEMPTY_INTENT_LABEL_STR: 261 (dropped 243)

=== Explosion sanity checks ===
Commit rows contributing: 261
Total counted intentions (sum k): 290
Rows with 2+ labels: 27 (10.3%)
Avg labels per row: 1.111

=== Observation 3.1 (Table 5) result ===
Total counted intentions (Table 5 Total k): 290
Top-2 intentions share: 57.6%

Table 5 (k and %)


,k,pct
intention,,
Expand test scope or capabilities,85.0,29.3
Introduce / strengthen CI-backed tests,82.0,28.3
Address performance or stability issues,41.0,14.1
Clean up or simplify CI / environment configuration,30.0,10.3
Automate or integrate release workflows,24.0,8.3
Migrate or modernise CI infrastructure,28.0,9.7
Total,290.0,100.0


                                                         k    pct
intention                                                        
Expand test scope or capabilities                     85.0   29.3
Introduce / strengthen CI-backed tests                82.0   28.3
Address performance or stability issues               41.0   14.1
Clean up or simplify CI / environment configuration   30.0   10.3
Automate or integrate release workflows               24.0    8.3
Migrate or modernise CI infrastructure                28.0    9.7
Total                                                290.0  100.0

DONE.


In [10]:
# ============================================================
# RQ3 Obs 3.2 from episodes
# (Chi-square + Cramér's V) + Table 6 (style skews)
# Prints in notebook only (no text log file).
# ============================================================

from pathlib import Path
import math
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency, fisher_exact

try:
    from IPython.display import display
except Exception:
    display = print  # fallback


# -----------------------------
# CONFIG
# -----------------------------
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output")
EPISODES_FILE = "2_All_episodes_with_intentions_subcat_tfidf.csv"

# Core filters to match Obs 3.2 / Table 6 wording
ONLY_EVENT_TYPE_ADDED = True
EVIDENCE_BACKED_TOTAL_SCORE_GT0 = True   # "non-zero text match"
REQUIRE_NONEMPTY_LABEL_STR = True        # selected labels only (label_str non-empty)

STYLE_RENAME = {
    "Emu_Community": "Community",
    "Emu_Custom": "Custom",
    "ThirdParty": "Third Party",
    "GMD": "GMD",
}

# Table 6 rows tested (BH/FDR applied across these 3)
TABLE6_TARGET_ROWS = [
    ("GMD", "Clean up or simplify CI / environment configuration"),
    ("Third Party", "Expand test scope or capabilities"),
    ("Community", "Migrate or modernise CI infrastructure"),
]


# -----------------------------
# Helpers
# -----------------------------
def resolve_path(base_dir: Path, fname: str) -> Path:
    p = base_dir / fname
    if p.exists():
        return p
    fb = Path("/mnt/data") / fname  # fallback for this chat sandbox
    if fb.exists():
        return fb
    raise FileNotFoundError(f"Could not find file:\n - {p}\n - {fb}")

def split_pipe(s):
    """Split strings like 'a || b' into ['a','b'] (stripped)."""
    if pd.isna(s):
        return []
    t = str(s).strip()
    if not t or t.lower() == "nan":
        return []
    return [p.strip() for p in t.split("||") if p.strip()]

def split_labels(label_str):
    """Explode multi-label strings into a list of labels."""
    if pd.isna(label_str):
        return []
    s = str(label_str).strip()
    if not s or s.lower() == "nan":
        return []
    if "||" in s:
        parts = [p.strip() for p in s.split("||")]
    elif ";" in s:
        parts = [p.strip() for p in s.split(";")]
    else:
        parts = [s]
    return [p for p in parts if p]

def normalize_style(style_raw: str) -> str:
    return STYLE_RENAME.get(style_raw, style_raw)

def bh_fdr(pvals):
    """Benjamini–Hochberg FDR correction (returns adjusted p-values in original order)."""
    pvals = np.asarray(pvals, dtype=float)
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    adj = np.empty(m, dtype=float)

    for i, p in enumerate(ranked, start=1):
        adj[i - 1] = p * m / i

    for i in range(m - 2, -1, -1):
        adj[i] = min(adj[i], adj[i + 1])

    adj = np.clip(adj, 0, 1)

    out = np.empty(m, dtype=float)
    out[order] = adj
    return out

def phi_from_2x2(table_2x2: np.ndarray) -> float:
    chi2_2, _, _, _ = chi2_contingency(table_2x2, correction=False)
    return float(math.sqrt(chi2_2 / table_2x2.sum()))


# -----------------------------
# RUN
# -----------------------------
episodes_path = resolve_path(BASE_DIR, EPISODES_FILE)
print("Episodes CSV:", episodes_path)

ep = pd.read_csv(episodes_path)
print("Loaded episode rows:", len(ep))

required_cols = [
    "env_styles", "boundary_event_types",
    "start_label_str", "start_total_score",
    "end_label_str", "end_total_score"
]
missing = [c for c in required_cols if c not in ep.columns]
if missing:
    raise KeyError(f"Missing required columns in episodes CSV: {missing}")

# Create per-boundary-event rows (start boundary = pos 0, end boundary = pos 1)
event_rows = []
for i, r in ep.iterrows():
    styles = split_pipe(r.get("env_styles", ""))
    types  = split_pipe(r.get("boundary_event_types", ""))

    for pos, ev_type in enumerate(types):
        ev_type_norm = str(ev_type).strip().lower()
        prefix = "start_" if pos == 0 else "end_"

        style_token = ""
        if styles:
            style_token = styles[pos] if pos < len(styles) else styles[-1]

        event_rows.append({
            "episode_row": i,
            "boundary_pos": pos,
            "event_type": ev_type_norm,
            "env_style": normalize_style(style_token),
            "label_str": r.get(prefix + "label_str", np.nan),
            "total_score": pd.to_numeric(r.get(prefix + "total_score", 0), errors="coerce"),
        })

events = pd.DataFrame(event_rows)
print("\nConstructed boundary-event rows:", len(events))
print("Boundary-event type counts:\n", events["event_type"].value_counts())

# Filters
f = events.copy()

if ONLY_EVENT_TYPE_ADDED:
    before = len(f)
    f = f[f["event_type"] == "added"].copy()
    print(f"\nAfter ONLY_EVENT_TYPE_ADDED: {len(f)} (dropped {before - len(f)})")

if EVIDENCE_BACKED_TOTAL_SCORE_GT0:
    before = len(f)
    f = f[f["total_score"].fillna(0) > 0].copy()
    print(f"After EVIDENCE_BACKED_TOTAL_SCORE_GT0: {len(f)} (dropped {before - len(f)})")

if REQUIRE_NONEMPTY_LABEL_STR:
    before = len(f)
    s = f["label_str"].astype(str).str.strip()
    f = f[s.ne("") & s.str.lower().ne("nan")].copy()
    print(f"After REQUIRE_NONEMPTY_LABEL_STR: {len(f)} (dropped {before - len(f)})")

# Explode label_str -> one row per detected intention
long_rows = []
for _, r in f.iterrows():
    for lab in split_labels(r["label_str"]):
        long_rows.append({"env_style": r["env_style"], "intention": lab})

long_df = pd.DataFrame(long_rows)
if long_df.empty:
    raise RuntimeError("No intentions found after filters + explode. Check your filters / columns.")

print("\n=== Sanity checks ===")
print("Boundary events contributing (after filters):", len(f))
print("Total counted intentions (N):", len(long_df))
print("Intentions per style (n):\n", long_df["env_style"].value_counts())

# -----------------------------
# Observation 3.2: Chi-square + Cramér's V
# -----------------------------
ct = pd.crosstab(long_df["env_style"], long_df["intention"])
chi2, p_value, dof, expected = chi2_contingency(ct)
N = ct.values.sum()
V = math.sqrt(chi2 / (N * (min(ct.shape) - 1)))

print("\n=== Observation 3.2 (Chi-square test of independence) ===")
print(f"Contingency table shape: {ct.shape} (styles x intentions)")
print(f"Total intentions (N): {N}")
print(f"chi2={chi2:.4f}, dof={dof}, p={p_value:.6g}, Cramér's V={V:.3f}")
display(ct)

# -----------------------------
# Table 6: Style-specific skews (paper rows)
# Fisher exact (two-sided) + phi + BH(FDR) across these 3 tests
# -----------------------------
total_by_style = long_df["env_style"].value_counts().to_dict()
total_all = int(len(long_df))

table6_rows = []
raw_pvals = []

for style, intention in TABLE6_TARGET_ROWS:
    n_style = int(total_by_style.get(style, 0))
    k_style = int(((long_df["env_style"] == style) & (long_df["intention"] == intention)).sum())

    n_other = total_all - n_style
    k_other = int((long_df["intention"] == intention).sum() - k_style)

    table = np.array([[k_style, n_style - k_style],
                      [k_other, n_other - k_other]], dtype=int)

    odds, p_fisher = fisher_exact(table, alternative="two-sided")
    phi = phi_from_2x2(table)

    raw_pvals.append(p_fisher)
    table6_rows.append({
        "Style": style,
        "Intention": intention,
        "Style k/n": f"{k_style}/{n_style} ({(k_style/n_style*100 if n_style else 0):.1f})",
        "Others k/n": f"{k_other}/{n_other} ({(k_other/n_other*100 if n_other else 0):.1f})",
        "p_raw (Fisher)": p_fisher,
        "phi": phi,
    })

p_fdr = bh_fdr(raw_pvals)
for i in range(len(table6_rows)):
    table6_rows[i]["p_FDR (BH over these 3)"] = float(p_fdr[i])

table6 = pd.DataFrame(table6_rows)[
    ["Style", "Intention", "Style k/n", "Others k/n", "p_raw (Fisher)", "p_FDR (BH over these 3)", "phi"]
].copy()

# Pretty formatting
table6["p_raw (Fisher)"] = table6["p_raw (Fisher)"].map(lambda x: f"{x:.3g}")
table6["p_FDR (BH over these 3)"] = table6["p_FDR (BH over these 3)"].map(lambda x: f"{x:.3g}")
table6["phi"] = table6["phi"].map(lambda x: f"{x:.3f}")

print("\n=== Table 6 (Style-specific intention skews) ===")
print("Notes: k counts detected intentions (not commits); n is total intentions within each group.")
display(table6)


Episodes CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\2_All_episodes_with_intentions_subcat_tfidf.csv
Loaded episode rows: 535

Constructed boundary-event rows: 636
Boundary-event type counts:
 event_type
added      600
removed     36
Name: count, dtype: int64

After ONLY_EVENT_TYPE_ADDED: 600 (dropped 36)
After EVIDENCE_BACKED_TOTAL_SCORE_GT0: 340 (dropped 260)
After REQUIRE_NONEMPTY_LABEL_STR: 325 (dropped 15)

=== Sanity checks ===
Boundary events contributing (after filters): 325
Total counted intentions (N): 359
Intentions per style (n):
 env_style
Community      220
Custom          79
Third Party     34
GMD             26
Name: count, dtype: int64

=== Observation 3.2 (Chi-square test of independence) ===
Contingency table shape: (4, 6) (styles x intentions)
Total intentions (N): 359
chi2=51.4016, dof=15, p=7.09086e-06, Cramér's V=0.218


intention,Address performance or stability issues,Automate or integrate release workflows,Clean up or simplify CI / environment configuration,Expand test scope or capabilities,Introduce / strengthen CI-backed tests,Migrate or modernise CI infrastructure
env_style,,,,,,
Community,31,11,27,49,65,37
Custom,14,10,3,23,26,3
GMD,3,0,7,12,4,0
Third Party,4,5,2,17,4,2



=== Table 6 (Style-specific intention skews) ===
Notes: k counts detected intentions (not commits); n is total intentions within each group.


,Style,Intention,Style k/n,Others k/n,p_raw (Fisher),p_FDR (BH over these 3),phi
0,GMD,Clean up or simplify CI / environment configur...,7/26 (26.9),32/333 (9.6),0.0144,0.0144,0.144
1,Third Party,Expand test scope or capabilities,17/34 (50.0),84/325 (25.8),0.00463,0.00695,0.157
2,Community,Migrate or modernise CI infrastructure,37/220 (16.8),5/139 (3.6),7.8e-05,0.000234,0.200


In [9]:
# ============================================================
# Approach (A) Intention Extraction Coverage Stats (ALL COMMITS)
# Source: 1_All_Commits_PR_Msg_Iss_with_intentions_subcat_tfidf.csv
# ============================================================

from pathlib import Path
import pandas as pd

# -----------------------------
# CONFIG
# -----------------------------
BASE_DIR = Path(r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output")
COMMITS_FILE = "1_All_Commits_PR_Msg_Iss_with_intentions_subcat_tfidf.csv"

# -----------------------------
# Helpers
# -----------------------------
def resolve_path(base_dir: Path, fname: str) -> Path:
    p = base_dir / fname
    if p.exists():
        return p
    # fallback for this chat sandbox
    fb = Path("/mnt/data") / fname
    if fb.exists():
        return fb
    raise FileNotFoundError(f"Could not find file:\n - {p}\n - {fb}")

def pct(n, d) -> float:
    return 0.0 if d == 0 else (100.0 * n / d)

def split_labels(s):
    if pd.isna(s):
        return []
    t = str(s).strip()
    if not t or t.lower() == "nan":
        return []
    return [p.strip() for p in t.split("||") if p.strip()]

# -----------------------------
# RUN
# -----------------------------
path = resolve_path(BASE_DIR, COMMITS_FILE)
print("Commits CSV:", path)

df = pd.read_csv(path)

required = ["qa_issue", "intent_total_score", "intent_label_str", "intent_confidence_level"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}")

total_boundary_commits = len(df)

# Exclude QA cases (qa_issue not null)
nonqa = df[df["qa_issue"].isna()].copy()
n_nonqa = len(nonqa)

# Evidence-backed matches
evidence = nonqa[nonqa["intent_total_score"].fillna(0) > 0].copy()
n_evidence = len(evidence)

# Final labeled (selected labels)
s = nonqa["intent_label_str"].astype(str).str.strip()
labeled = nonqa[s.ne("") & s.str.lower().ne("nan")].copy()
n_labeled = len(labeled)

# Confidence breakdown among labeled
conf_counts = (
    labeled["intent_confidence_level"]
    .astype(str)
    .str.upper()
    .value_counts()
    .to_dict()
)
n_high = int(conf_counts.get("HIGH", 0))
n_med  = int(conf_counts.get("MEDIUM", 0))
n_low  = int(conf_counts.get("LOW", 0))

# Multi-label counts among labeled (based on intent_label_str)
label_counts = labeled["intent_label_str"].apply(lambda x: len(split_labels(x)))
n_1 = int((label_counts == 1).sum())
n_2 = int((label_counts == 2).sum())
n_3plus = int((label_counts >= 3).sum())  # robust
n_multi = int((label_counts >= 2).sum())

# -----------------------------
# PRINT RESULTS
# -----------------------------
print("\n=== Summary ===")
print(f"Total boundary commits (rows in file): {total_boundary_commits}")
print(f"Exclude QA flagged: {total_boundary_commits - n_nonqa}")
print(f"Commits for intention analysis: {n_nonqa}")

print("\n=== Coverage ===")
print(f"Evidence-backed match (intent_total_score > 0): {n_evidence}/{n_nonqa} ({pct(n_evidence, n_nonqa):.1f}%)")
print(f"Retain >=1 final intention label (non-empty intent_label_str): {n_labeled}/{n_nonqa} ({pct(n_labeled, n_nonqa):.1f}%)")

print("\n=== Confidence among labeled commits ===")
print(f"HIGH:   {n_high}/{n_labeled} ({pct(n_high, n_labeled):.1f}%)")
print(f"MEDIUM: {n_med}/{n_labeled} ({pct(n_med, n_labeled):.1f}%)")
print(f"LOW:    {n_low}/{n_labeled} ({pct(n_low, n_labeled):.1f}%)")

print("\n=== Multi-labeling among labeled commits (intent_label_str split on '||') ===")
print(f"1 label:  {n_1}/{n_labeled} ({pct(n_1, n_labeled):.1f}%)")
print(f"2 labels: {n_2}/{n_labeled} ({pct(n_2, n_labeled):.1f}%)")
print(f"3+ labels:{n_3plus}/{n_labeled} ({pct(n_3plus, n_labeled):.1f}%)")
print(f"2+ labels (multi-label): {n_multi}/{n_labeled} ({pct(n_multi, n_labeled):.1f}%)")

print("\n=== Paper-ready sentence (copy/paste) ===")
print(
    f"From {total_boundary_commits} unique boundary commits (RQ2), we exclude {total_boundary_commits - n_nonqa} flagged QA cases "
    f"(no execution environment detected at cutoff), leaving {n_nonqa} commits for intention analysis. "
    f"The model finds at least one evidence-backed match for {n_evidence}/{n_nonqa} commits ({pct(n_evidence, n_nonqa):.1f}\\%); "
    f"after Subcat selection and gating (final output in \\texttt{{intent\\_label\\_str}}), {n_labeled}/{n_nonqa} commits "
    f"({pct(n_labeled, n_nonqa):.1f}\\%) retain at least one intention label. Among labeled commits, {n_high} "
    f"({pct(n_high, n_labeled):.1f}\\%) are \\textsc{{HIGH}} confidence, {n_med} ({pct(n_med, n_labeled):.1f}\\%) "
    f"\\textsc{{MEDIUM}}, and {n_low} ({pct(n_low, n_labeled):.1f}\\%) \\textsc{{LOW}}; {n_multi}/{n_labeled} "
    f"({pct(n_multi, n_labeled):.1f}\\%) receive multiple labels ({n_2} receive two labels; {n_3plus} receive three labels)."
)


Commits CSV: C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\ICST2026\3 - RQ3\Part_1_Intention\Output\1_All_Commits_PR_Msg_Iss_with_intentions_subcat_tfidf.csv

=== Summary ===
Total boundary commits (rows in file): 535
Exclude QA flagged: 11
Commits for intention analysis: 524

=== Coverage ===
Evidence-backed match (intent_total_score > 0): 288/524 (55.0%)
Retain >=1 final intention label (non-empty intent_label_str): 276/524 (52.7%)

=== Confidence among labeled commits ===
HIGH:   132/276 (47.8%)
MEDIUM: 98/276 (35.5%)
LOW:    46/276 (16.7%)

=== Multi-labeling among labeled commits (intent_label_str split on '||') ===
1 label:  247/276 (89.5%)
2 labels: 27/276 (9.8%)
3+ labels:2/276 (0.7%)
2+ labels (multi-label): 29/276 (10.5%)

=== Paper-ready sentence (copy/paste) ===
From 535 unique boundary commits (RQ2), we exclude 11 flagged QA cases (no execution environment detected at cutoff), leaving 524 commits for intention analysis. The model finds at least one evide